
# PCA en casos de texto

_________________________

In [ ]:
# Dependencies
import re
import os
import sys
import io
import pandas as pd
import zipfile
import json
import numpy as np
import ast
from sklearn.decomposition import PCA
import plotly.express as px
#
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Dataset Original:
# https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/data?select=Combined+Data.csv

# Usaremos solo el 5% del dataset, la selección fue leatoria con:
# df_combined_sample = df_combined.sample(frac=0.05, random_state=42)

In [ ]:
# Path to the zip file
zip_file_path = 'Data_embedding.csv.zip'
# Name of the CSV file inside the zip
csv_file_name = 'Data_embedding.csv'

# Extract the CSV file from the zip archive
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extract(csv_file_name, path='.') # Extract to current directory

# Load the data from Data_embedding.csv into a pandas DataFrame
df = pd.read_csv(csv_file_name)
df.head()

In [ ]:
# Display the frequency table of the 'status' column
status_counts = df['status'].value_counts()
display(status_counts)

In [ ]:
# --- 1) Preparar X (matriz de embeddings) ---
cols_req = ["embedding", "status", "statement"]
mask = df[cols_req].notna().all(axis=1)
df_ = df.loc[mask, cols_req].copy()


In [ ]:
# Convierte la columna embedding (lista/array por fila en formato string) en una matriz 2D (n_muestras x dim)
# Usa ast.literal_eval para convertir las strings de lista en listas de Python
X = np.array([ast.literal_eval(e) for e in df_["embedding"].tolist()], dtype=float)

#assert X.ndim == 2, "Cada 'embedding' debe ser una lista/array de igual longitud."
#assert np.isfinite(X).all(), "Hay valores no finitos (NaN/Inf) en los embeddings."

In [ ]:
# --- 2) PCA a 3 componentes ---
# Nota: PCA centra automáticamente las features (no escala a var=1).
pca = PCA(n_components=3, random_state=0)
X_pca = pca.fit_transform(X)  # shape: (n_muestras, 3)


In [ ]:
# --- 3) DataFrame para graficar ---
df_plot = pd.DataFrame(
    X_pca, columns=["PC1", "PC2", "PC3"], index=df_.index
).assign(
    status=df_["status"].astype(str),
    statement=df_["statement"].astype(str)
)

In [ ]:
# Títulos con varianza explicada
var = pca.explained_variance_ratio_
axis_titles = {
    "x": f"PC1 ({var[0]:.1%})",
    "y": f"PC2 ({var[1]:.1%})",
    "z": f"PC3 ({var[2]:.1%})",
}
axis_titles

In [ ]:
# --- 4) Gráfica 3D interactiva con Plotly ---
fig = px.scatter_3d(
    df_plot, x="PC1", y="PC2", z="PC3",
    color="status",
    hover_data={"statement": True, "status": True, "PC1": ':.3f', "PC2": ':.3f', "PC3": ':.3f'},
    opacity=0.85,
    width=900, height=650,
    title="PCA de embeddings (3D)"
)

fig.update_layout(
    scene=dict(
        xaxis_title=axis_titles["x"],
        yaxis_title=axis_titles["y"],
        zaxis_title=axis_titles["z"],
    ),
    legend_title_text="status",
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()